In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .getOrCreate()
    )
    return spark


trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
parquet_root = trec_root / "data/enwiki/parquet"
spark = get_spark()

page = spark.read.parquet((parquet_root / "page").as_posix())
nodes = spark.read.parquet((dataset_root / "graph/v1/nodes"))
pagerank = spark.read.parquet((dataset_root / "centrality/v1/pagerank.parquet"))
hits = spark.read.parquet((dataset_root / "centrality/v1/hits.parquet"))

In [ ]:
df = (
    nodes.join(
        page.select("page_id", "page_title"),
        on="page_id",
        how="inner",
    )
    .join(pagerank, on="node_id", how="inner")
    .join(hits, on="node_id", how="inner")
)